# Phase 2 — TxGNN-architecture model (PyTorch Geometric reimplementation)

**This is not the original authors' pretrained TxGNN checkpoint.** That pip package (`TxGNN`, DGL 0.5.2-based) fails to install on current Colab Python (Phase 0). This notebook reimplements TxGNN's core idea directly in PyTorch Geometric:

- A relational GNN encoder over the full heterogeneous PrimeKG (all node/edge types), producing one embedding per node.
- A **two-phase** training procedure matching TxGNN's design: (A) self-supervised pretraining across many relation types, then (B) fine-tuning on the actual task (drug–disease indication / contraindication link prediction).
- A dot-product decoder — deliberately the *same* decoder used in the HGT/HAN baselines (Phase 3), so the comparison isolates the encoder architecture rather than decoder differences. The original TxGNN's own decoder differs in detail; this is a documented simplification for a fair, apples-to-apples comparison.

Every metric this notebook produces gets labelled **"TxGNN-architecture reimplementation (PyG), this study"** everywhere it's reported — not attributed to the original authors' numbers.

Splits (random + zero-shot) are constructed here from scratch, since we no longer have TxGNN's `TxData.prepare_split`. Methodology is documented per split below. Seed fixed at 42 throughout.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT_DIR = '/content/drive/MyDrive/txgnn_comparison'
for sub in ['data', 'checkpoints', 'results']:
    os.makedirs(f'{PROJECT_DIR}/{sub}', exist_ok=True)
kg_path = f'{PROJECT_DIR}/data/kg.csv'
assert os.path.exists(kg_path), 'Run Phase 1 first — kg.csv not found on Drive.'
print('Using', kg_path)

Mounted at /content/drive
Using /content/drive/MyDrive/txgnn_comparison/data/kg.csv


In [2]:
import pandas as pd, numpy as np, torch, json, time, random

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

kg = pd.read_csv(kg_path, low_memory=False)
print('Loaded kg.csv:', kg.shape)

Device: cuda
Loaded kg.csv: (8100498, 12)


In [3]:
# Find the real indication/contraindication relation names — derived from the
# actual data at runtime, not hardcoded, since we haven't independently
# verified PrimeKG's exact relation strings in this session.
all_relations = kg['relation'].unique().tolist()
indication_rels = [r for r in all_relations if 'indicat' in r.lower() and 'contra' not in r.lower()]
contraindication_rels = [r for r in all_relations if 'contraindicat' in r.lower()]
print('All relations:', all_relations)
print('Matched indication relations:', indication_rels)
print('Matched contraindication relations:', contraindication_rels)
assert len(indication_rels) > 0, 'No indication relation found — inspect all_relations above and fix the filter.'
assert len(contraindication_rels) > 0, 'No contraindication relation found — inspect all_relations above and fix the filter.'

All relations: ['protein_protein', 'drug_protein', 'contraindication', 'indication', 'off-label use', 'drug_drug', 'phenotype_protein', 'phenotype_phenotype', 'disease_phenotype_negative', 'disease_phenotype_positive', 'disease_protein', 'disease_disease', 'drug_effect', 'bioprocess_bioprocess', 'molfunc_molfunc', 'cellcomp_cellcomp', 'molfunc_protein', 'cellcomp_protein', 'bioprocess_protein', 'exposure_protein', 'exposure_disease', 'exposure_exposure', 'exposure_bioprocess', 'exposure_molfunc', 'exposure_cellcomp', 'pathway_pathway', 'pathway_protein', 'anatomy_anatomy', 'anatomy_protein_present', 'anatomy_protein_absent']
Matched indication relations: ['indication']
Matched contraindication relations: ['contraindication']


In [4]:
# Build global node table and per-type local index mapping
nodes_x = kg[['x_index', 'x_id', 'x_type', 'x_name']].rename(columns=lambda c: c[2:])
nodes_y = kg[['y_index', 'y_id', 'y_type', 'y_name']].rename(columns=lambda c: c[2:])
all_nodes = pd.concat([nodes_x, nodes_y]).drop_duplicates(subset=['index']).sort_values('index').reset_index(drop=True)

node_types = sorted(all_nodes['type'].unique())
local_idx = {}      # global_index -> (type, local_idx)
type_counts = {}
for t in node_types:
    sub = all_nodes[all_nodes['type'] == t].sort_values('index')
    type_counts[t] = len(sub)
    for local_i, global_i in enumerate(sub['index'].values):
        local_idx[global_i] = (t, local_i)

print('Node types and counts:', type_counts)

Node types and counts: {'anatomy': 14035, 'biological_process': 28642, 'cellular_component': 4176, 'disease': 17080, 'drug': 7957, 'effect/phenotype': 15311, 'exposure': 818, 'gene/protein': 27671, 'molecular_function': 11169, 'pathway': 2516}


In [9]:
# 0.2 Install PyTorch Geometric and shared dependencies (no DGL, no TxGNN pip package)
!pip install -q torch-geometric transformers scikit-learn pandas numpy matplotlib seaborn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 36.8 MB/s eta 0:00:0000:01


In [10]:
# Build HeteroData graph: one learnable embedding table per node type
# (PrimeKG ships no node features, so embeddings are learned end-to-end).
import torch
import torch_geometric
from torch_geometric.data import HeteroData

HIDDEN = 64  # matches HGT/HAN baseline hidden_channels for a fair comparison
data = HeteroData()
for t in node_types:
    data[t].num_nodes = type_counts[t]

edge_type_groups = kg.groupby(['x_type', 'relation', 'y_type'])
edge_index_by_type = {}
for (src_t, rel, dst_t), grp in edge_type_groups:
    src_local = grp['x_index'].map(lambda g: local_idx[g][1]).values
    dst_local = grp['y_index'].map(lambda g: local_idx[g][1]).values
    ei = torch.tensor(np.stack([src_local, dst_local]), dtype=torch.long)
    data[(src_t, rel, dst_t)].edge_index = ei
    edge_index_by_type[(src_t, rel, dst_t)] = ei

print('Built', len(edge_index_by_type), 'edge types in HeteroData')
print(data)

Built 50 edge types in HeteroData
HeteroData(
  anatomy={ num_nodes=14035 },
  biological_process={ num_nodes=28642 },
  cellular_component={ num_nodes=4176 },
  disease={ num_nodes=17080 },
  drug={ num_nodes=7957 },
  effect/phenotype={ num_nodes=15311 },
  exposure={ num_nodes=818 },
  gene/protein={ num_nodes=27671 },
  molecular_function={ num_nodes=11169 },
  pathway={ num_nodes=2516 },
  (anatomy, anatomy_anatomy, anatomy)={ edge_index=[2, 28064] },
  (anatomy, anatomy_protein_absent, gene/protein)={ edge_index=[2, 19887] },
  (anatomy, anatomy_protein_present, gene/protein)={ edge_index=[2, 1518203] },
  (biological_process, bioprocess_bioprocess, biological_process)={ edge_index=[2, 105772] },
  (biological_process, bioprocess_protein, gene/protein)={ edge_index=[2, 144805] },
  (biological_process, exposure_bioprocess, exposure)={ edge_index=[2, 1625] },
  (cellular_component, cellcomp_cellcomp, cellular_component)={ edge_index=[2, 9690] },
  (cellular_component, cellcomp_pro

In [11]:
# Drug-disease positive pairs for the two target relations
def pairs_for(rel_list):
    sub = kg[kg['relation'].isin(rel_list)]
    # normalise direction to (drug_local, disease_local) regardless of x/y order
    pairs = []
    for _, row in sub.iterrows():
        if row['x_type'] == 'drug' and row['y_type'] == 'disease':
            pairs.append((local_idx[row['x_index']][1], local_idx[row['y_index']][1]))
        elif row['x_type'] == 'disease' and row['y_type'] == 'drug':
            pairs.append((local_idx[row['y_index']][1], local_idx[row['x_index']][1]))
    return np.array(sorted(set(pairs)))

indication_pairs = pairs_for(indication_rels)
contraindication_pairs = pairs_for(contraindication_rels)
print('Indication pairs (drug, disease):', len(indication_pairs))
print('Contraindication pairs (drug, disease):', len(contraindication_pairs))
n_drug, n_disease = type_counts['drug'], type_counts['disease']

Indication pairs (drug, disease): 9388
Contraindication pairs (drug, disease): 30675


In [12]:
# Split construction — methodology for THIS study (not TxGNN's original splits)
rng = np.random.RandomState(SEED)

def random_split(pairs, train=0.7, valid=0.15):
    idx = rng.permutation(len(pairs))
    n_train = int(train * len(pairs))
    n_valid = int(valid * len(pairs))
    return pairs[idx[:n_train]], pairs[idx[n_train:n_train+n_valid]], pairs[idx[n_train+n_valid:]]

def zero_shot_split(pairs, n_disease, holdout_frac=0.2):
    diseases_with_edges = np.unique(pairs[:, 1])
    n_holdout = int(holdout_frac * len(diseases_with_edges))
    holdout_diseases = set(rng.choice(diseases_with_edges, size=n_holdout, replace=False))
    is_test = np.array([d in holdout_diseases for d in pairs[:, 1]])
    test = pairs[is_test]
    remaining = pairs[~is_test]
    idx = rng.permutation(len(remaining))
    n_valid = int(0.15 * len(remaining))
    valid = remaining[idx[:n_valid]]
    train = remaining[idx[n_valid:]]
    return train, valid, test, holdout_diseases

ind_train_r, ind_valid_r, ind_test_r = random_split(indication_pairs)
contra_train_r, contra_valid_r, contra_test_r = random_split(contraindication_pairs)
ind_train_z, ind_valid_z, ind_test_z, ind_holdout = zero_shot_split(indication_pairs, n_disease)
contra_train_z, contra_valid_z, contra_test_z, contra_holdout = zero_shot_split(contraindication_pairs, n_disease)

print('Random split  — indication train/valid/test:', len(ind_train_r), len(ind_valid_r), len(ind_test_r))
print('Random split  — contraindication train/valid/test:', len(contra_train_r), len(contra_valid_r), len(contra_test_r))
print('Zero-shot split — indication train/valid/test:', len(ind_train_z), len(ind_valid_z), len(ind_test_z), '| held-out diseases:', len(ind_holdout))
print('Zero-shot split — contraindication train/valid/test:', len(contra_train_z), len(contra_valid_z), len(contra_test_z), '| held-out diseases:', len(contra_holdout))

Random split  — indication train/valid/test: 6571 1408 1409
Random split  — contraindication train/valid/test: 21472 4601 4602
Zero-shot split — indication train/valid/test: 6211 1095 2082 | held-out diseases: 272
Zero-shot split — contraindication train/valid/test: 21295 3757 5623 | held-out diseases: 239


In [13]:
# Persist splits to Drive so the Phase 3 HGT/HAN baselines train and test on
# the exact same drug-disease pairs (required for a fair comparison).
np.savez(f'{PROJECT_DIR}/data/splits.npz',
         ind_train_r=ind_train_r, ind_valid_r=ind_valid_r, ind_test_r=ind_test_r,
         contra_train_r=contra_train_r, contra_valid_r=contra_valid_r, contra_test_r=contra_test_r,
         ind_train_z=ind_train_z, ind_valid_z=ind_valid_z, ind_test_z=ind_test_z,
         contra_train_z=contra_train_z, contra_valid_z=contra_valid_z, contra_test_z=contra_test_z,
         n_drug=n_drug, n_disease=n_disease)
print('Saved splits to', f'{PROJECT_DIR}/data/splits.npz')

Saved splits to /content/drive/MyDrive/txgnn_comparison/data/splits.npz


In [14]:
# Model: heterogeneous relational encoder + dot-product decoder
import torch.nn as nn
from torch_geometric.nn import HeteroConv, SAGEConv

class RelationalEncoder(nn.Module):
    def __init__(self, node_types, edge_types, type_counts, hidden=HIDDEN, layers=2):
        super().__init__()
        self.embed = nn.ModuleDict({t: nn.Embedding(type_counts[t], hidden) for t in node_types})
        self.convs = nn.ModuleList([
            HeteroConv({et: SAGEConv(hidden, hidden) for et in edge_types}, aggr='mean')
            for _ in range(layers)
        ])

    def forward(self, edge_index_dict):
        x_dict = {t: emb.weight for t, emb in self.embed.items()}
        for conv in self.convs:
            x_dict = conv(x_dict, edge_index_dict)
            x_dict = {k: v.relu() for k, v in x_dict.items()}
        return x_dict

def decode(emb_drug, emb_disease, drug_idx, disease_idx):
    return (emb_drug[drug_idx] * emb_disease[disease_idx]).sum(dim=-1)

edge_types = list(edge_index_by_type.keys())
model = RelationalEncoder(node_types, edge_types, type_counts).to(device)
edge_index_dict = {k: v.to(device) for k, v in edge_index_by_type.items()}
param_count = sum(p.numel() for p in model.parameters())
print('Param count:', param_count)

Param count: 9105600


In [15]:
# Training utilities
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, roc_auc_score

def sample_negatives(pos_pairs, n_drug, n_disease, k=1):
    pos_set = set(map(tuple, pos_pairs))
    neg = []
    while len(neg) < len(pos_pairs) * k:
        d, s = rng.randint(0, n_drug), rng.randint(0, n_disease)
        if (d, s) not in pos_set:
            neg.append((d, s))
    return np.array(neg)

def bce_link_loss(z_dict, pos_pairs, src_type, dst_type, n_src, n_dst):
    neg_pairs = sample_negatives(pos_pairs, n_src, n_dst)
    pos_idx = torch.tensor(pos_pairs, dtype=torch.long, device=device)
    neg_idx = torch.tensor(neg_pairs, dtype=torch.long, device=device)
    pos_score = decode(z_dict[src_type], z_dict[dst_type], pos_idx[:, 0], pos_idx[:, 1])
    neg_score = decode(z_dict[src_type], z_dict[dst_type], neg_idx[:, 0], neg_idx[:, 1])
    scores = torch.cat([pos_score, neg_score])
    labels = torch.cat([torch.ones_like(pos_score), torch.zeros_like(neg_score)])
    return F.binary_cross_entropy_with_logits(scores, labels)

def evaluate(z_dict, pos_pairs, n_drug, n_disease):
    neg_pairs = sample_negatives(pos_pairs, n_drug, n_disease)
    pos_idx = torch.tensor(pos_pairs, dtype=torch.long, device=device)
    neg_idx = torch.tensor(neg_pairs, dtype=torch.long, device=device)
    with torch.no_grad():
        pos_score = torch.sigmoid(decode(z_dict['drug'], z_dict['disease'], pos_idx[:, 0], pos_idx[:, 1])).cpu().numpy()
        neg_score = torch.sigmoid(decode(z_dict['drug'], z_dict['disease'], neg_idx[:, 0], neg_idx[:, 1])).cpu().numpy()
    scores = np.concatenate([pos_score, neg_score])
    labels = np.concatenate([np.ones(len(pos_score)), np.zeros(len(neg_score))])
    return {'AUPRC': float(average_precision_score(labels, scores)), 'AUROC': float(roc_auc_score(labels, scores))}

## Phase A — self-supervised pretraining across the full heterogeneous graph

Mirrors TxGNN's pretraining stage: the encoder learns from the structure of the *entire* PrimeKG (all relation types), not just drug-disease links.

In [16]:
PRETRAIN_EPOCHS = 50
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
pretrain_log = []

# Sample a handful of relation types per epoch to keep pretraining tractable
# given PrimeKG's 8M+ edges (notably anatomy_protein_present and drug_drug).
pretrain_rel_pool = [et for et in edge_types if et[1] not in indication_rels + contraindication_rels]

t0 = time.time()
for epoch in range(PRETRAIN_EPOCHS):
    model.train()
    z_dict = model(edge_index_dict)
    rel = pretrain_rel_pool[epoch % len(pretrain_rel_pool)]
    src_t, _, dst_t = rel
    ei = edge_index_by_type[rel].numpy().T
    sample = ei[rng.choice(len(ei), size=min(20000, len(ei)), replace=False)]
    loss = bce_link_loss(z_dict, sample, src_t, dst_t, type_counts[src_t], type_counts[dst_t])
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    pretrain_log.append({'epoch': epoch, 'relation': str(rel), 'loss': float(loss.item())})
    if epoch % 10 == 0:
        print(f'[pretrain] epoch {epoch} rel={rel} loss={loss.item():.4f}')

pretrain_time_min = (time.time() - t0) / 60
pd.DataFrame(pretrain_log).to_csv(f'{PROJECT_DIR}/results/txgnn_pyg_pretrain_log.csv', index=False)
print('Pretraining done in', round(pretrain_time_min, 2), 'min')

[pretrain] epoch 0 rel=('anatomy', 'anatomy_anatomy', 'anatomy') loss=0.7035
[pretrain] epoch 10 rel=('disease', 'disease_phenotype_negative', 'effect/phenotype') loss=0.6926
[pretrain] epoch 20 rel=('effect/phenotype', 'disease_phenotype_positive', 'disease') loss=0.6877
[pretrain] epoch 30 rel=('gene/protein', 'anatomy_protein_absent', 'anatomy') loss=0.6719
[pretrain] epoch 40 rel=('gene/protein', 'protein_protein', 'gene/protein') loss=0.6890
Pretraining done in 0.23 min


## Phase B — fine-tuning + evaluation on a given split

Reusable for both the random split and the zero-shot split.

In [17]:
def finetune_and_eval(split_name, ind_train, ind_valid, ind_test, contra_train, contra_valid, contra_test, n_epoch=200, lr=5e-4):
    ft_model = RelationalEncoder(node_types, edge_types, type_counts).to(device)
    ft_model.load_state_dict(model.state_dict())  # start from pretrained weights
    opt = torch.optim.Adam(ft_model.parameters(), lr=lr)
    log = []
    t0 = time.time()
    for epoch in range(n_epoch):
        ft_model.train()
        z_dict = ft_model(edge_index_dict)
        loss_ind = bce_link_loss(z_dict, ind_train, 'drug', 'disease', n_drug, n_disease)
        loss_contra = bce_link_loss(z_dict, contra_train, 'drug', 'disease', n_drug, n_disease)
        loss = loss_ind + loss_contra
        opt.zero_grad(); loss.backward(); opt.step()
        log.append({'epoch': epoch, 'loss_indication': float(loss_ind.item()), 'loss_contraindication': float(loss_contra.item())})
        if epoch % 25 == 0:
            ft_model.eval()
            with torch.no_grad():
                z_eval = ft_model(edge_index_dict)
            valid_metrics = evaluate(z_eval, ind_valid, n_drug, n_disease)
            print(f'[{split_name}] epoch {epoch} loss={loss.item():.4f} valid_indication_AUPRC={valid_metrics["AUPRC"]:.4f}')
    train_time_min = (time.time() - t0) / 60

    ft_model.eval()
    with torch.no_grad():
        z_final = ft_model(edge_index_dict)
    ind_metrics = evaluate(z_final, ind_test, n_drug, n_disease)
    contra_metrics = evaluate(z_final, contra_test, n_drug, n_disease)

    pd.DataFrame(log).to_csv(f'{PROJECT_DIR}/results/txgnn_pyg_{split_name}_training_log.csv', index=False)

    result = {
        'model': 'TxGNN-architecture reimplementation (PyG), this study',
        'split': split_name,
        'seed': SEED,
        'indication_AUPRC': ind_metrics['AUPRC'],
        'indication_AUROC': ind_metrics['AUROC'],
        'contraindication_AUPRC': contra_metrics['AUPRC'],
        'contraindication_AUROC': contra_metrics['AUROC'],
        'training_time_min': round(train_time_min, 2),
        'param_count': param_count,
        'notes': 'Not the original TxGNN checkpoint. PyG reimplementation: relational SAGEConv encoder pretrained on full PrimeKG, fine-tuned with dot-product decoder. Split constructed for this study, seed 42.'
    }
    with open(f'{PROJECT_DIR}/results/txgnn_pyg_{split_name}.json', 'w') as f:
        json.dump(result, f, indent=2)
    print(json.dumps(result, indent=2))
    return result

In [18]:
result_random = finetune_and_eval('random', ind_train_r, ind_valid_r, ind_test_r, contra_train_r, contra_valid_r, contra_test_r)

[random] epoch 0 loss=1.2949 valid_indication_AUPRC=0.9374
[random] epoch 25 loss=0.9938 valid_indication_AUPRC=0.9796
[random] epoch 50 loss=0.8642 valid_indication_AUPRC=0.9731
[random] epoch 75 loss=0.8253 valid_indication_AUPRC=0.9862
[random] epoch 100 loss=0.8071 valid_indication_AUPRC=0.9898
[random] epoch 125 loss=0.7952 valid_indication_AUPRC=0.9879
[random] epoch 150 loss=0.7865 valid_indication_AUPRC=0.9877
[random] epoch 175 loss=0.7760 valid_indication_AUPRC=0.9936
{
  "model": "TxGNN-architecture reimplementation (PyG), this study",
  "split": "random",
  "seed": 42,
  "indication_AUPRC": 0.9911400736950633,
  "indication_AUROC": 0.9934462678079324,
  "contraindication_AUPRC": 0.9933417011464862,
  "contraindication_AUROC": 0.9955717390224496,
  "training_time_min": 1.15,
  "param_count": 9105600,
  "notes": "Not the original TxGNN checkpoint. PyG reimplementation: relational SAGEConv encoder pretrained on full PrimeKG, fine-tuned with dot-product decoder. Split construct

In [19]:
result_zeroshot = finetune_and_eval('zero_shot', ind_train_z, ind_valid_z, ind_test_z, contra_train_z, contra_valid_z, contra_test_z)

[zero_shot] epoch 0 loss=1.2943 valid_indication_AUPRC=0.9512
[zero_shot] epoch 25 loss=0.9910 valid_indication_AUPRC=0.9768
[zero_shot] epoch 50 loss=0.8654 valid_indication_AUPRC=0.9881
[zero_shot] epoch 75 loss=0.8223 valid_indication_AUPRC=0.9836
[zero_shot] epoch 100 loss=0.8058 valid_indication_AUPRC=0.9929
[zero_shot] epoch 125 loss=0.7887 valid_indication_AUPRC=0.9877
[zero_shot] epoch 150 loss=0.7862 valid_indication_AUPRC=0.9943
[zero_shot] epoch 175 loss=0.7762 valid_indication_AUPRC=0.9916
{
  "model": "TxGNN-architecture reimplementation (PyG), this study",
  "split": "zero_shot",
  "seed": 42,
  "indication_AUPRC": 0.9799452567291346,
  "indication_AUROC": 0.9859299000351579,
  "contraindication_AUPRC": 0.9831985991764324,
  "contraindication_AUROC": 0.9892129765173643,
  "training_time_min": 1.16,
  "param_count": 9105600,
  "notes": "Not the original TxGNN checkpoint. PyG reimplementation: relational SAGEConv encoder pretrained on full PrimeKG, fine-tuned with dot-produ

**Acceptance gate:** both `txgnn_pyg_random.json` and `txgnn_pyg_zero_shot.json` exist under `results/` on Drive, AUPRC/AUROC are finite numbers in [0, 1], and training logs are non-empty. Copy both result JSONs back into this repo's `results/` folder (or paste their contents here) before moving to Phase 3.